# Notebook 01 — Structure du catalogue Steam

## Objectif du notebook

Ce notebook constitue la première étape du projet.  
Son objectif est de **comprendre la structure du dataset Steam et de préparer les données pour les analyses exploratoires**.

Le dataset utilisé est un **fichier JSON semi-structuré**, ce qui signifie que certaines informations sont stockées dans des structures imbriquées ou sous forme de listes. Avant de réaliser des analyses de marché, il est donc nécessaire d’examiner et de transformer ces données.

## Dataset utilisé

Le dataset est disponible dans un bucket S3 :

s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json

## Enjeux techniques

Le dataset étant **semi-structuré**, plusieurs opérations sont nécessaires pour l’exploiter correctement :

- inspection du **schéma Spark**
- exploration des **structures imbriquées**
- manipulation des **tableaux et objets JSON**
- transformation de certaines colonnes

## Étapes du notebook

Les principales étapes de ce notebook sont les suivantes :

1. **Préparation des données**  
   Chargement du dataset JSON dans PySpark, exploration du schéma Spark et transformation des structures imbriquées afin de construire une table exploitable pour l’analyse.

2. **Création de dataframes auxiliaires**  
   Construction de tables spécifiques permettant d’analyser certaines dimensions du catalogue, notamment les **tags, les plateformes, les catégories et les éditeurs**.

3. **Analyse de la croissance du catalogue Steam**  
   Étude de l’évolution du nombre de jeux publiés et du nombre d’éditeurs actifs au fil du temps.

4. **Analyse des principales caractéristiques du catalogue**  
   Exploration de plusieurs dimensions structurantes du catalogue Steam :
   - structure des éditeurs
   - plateformes supportées
   - langues disponibles

5. **Analyse des types de jeux présents sur la plateforme**  
   Étude des **catégories de gameplay** et de la **distribution des genres**, afin de mieux comprendre la composition du catalogue et le poids des productions indépendantes.

Les observations issues de cette analyse descriptive serviront de base aux analyses plus approfondies réalisées dans le **Notebook 02**.

# 1. Préparation des données

### 1.1 Chargement des données

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.utils import IllegalArgumentException
from pyspark.sql.functions import col, map_from_arrays, array, lit, explode
from pyspark.sql.functions import count
from pyspark.sql.window import Window
import math
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [0]:
spark

In [0]:
df = spark.read.json(
"s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
)

display(df)

data,id
"List(10, List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP), 13990, Valve, 0, Action, https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513, 999, English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean, Counter-Strike, 5199, 10,000,000 .. 20,000,000, List(true, true, true), 201215, 999, Valve, 2000/11/1, 0, Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role., List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192), game, )",10
"List(1000000, List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud), 0, IndigoBlue Game Studio, 0, Action, Adventure, Indie, https://cdn.akamai.steamstatic.com/steam/apps/1000000/header.jpg?t=1655723048, 999, English, Korean, Simplified Chinese, ASCENXION, 5, 0 .. 20,000, List(false, false, true), 27, 999, PsychoFlux Entertainment, 2021/05/14, 0, ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, 

In [0]:
df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:
df.count()

55691

In [0]:
display(df.limit(20))

data,id
"List(10, List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP), 13990, Valve, 0, Action, https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513, 999, English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean, Counter-Strike, 5199, 10,000,000 .. 20,000,000, List(true, true, true), 201215, 999, Valve, 2000/11/1, 0, Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role., List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192), game, )",10
"List(1000000, List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud), 0, IndigoBlue Game Studio, 0, Action, Adventure, Indie, https://cdn.akamai.steamstatic.com/steam/apps/1000000/header.jpg?t=1655723048, 999, English, Korean, Simplified Chinese, ASCENXION, 5, 0 .. 20,000, List(false, false, true), 27, 999, PsychoFlux Entertainment, 2021/05/14, 0, ASCENXION is a 2D shoot 'em up game where you explore the field to progress. Players must overcome puzzles, traps, elite units, boss fights, and other various obstacles while navigating the field. Grow stronger through rewards earned, 

### 1.2 Flatten du champ "data"

Les données sont contenues dans une structure "data". Nous aplatissons donc le dataframe en extrayant toutes les données de "Data" et en supprimant ce niveau.

In [0]:
# Flattening the DataFrame
df = df.select("*", "data.*").drop("data")
# print the schema of the flattened DataFrame
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- appid: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- platforms: struct (nullable = true)
 |    |-- linux: boolean (nullable = true)
 |    |-- mac: boolean (nullable = true)
 |    |-- windows: boolean (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- tags: struct (nul

In [0]:
pd.set_option('display.max_columns', None)
df.limit(5).toPandas()


,id,appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website
0,10,10,"[Multi-player, Valve Anti-Cheat enabled, Onlin...",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/...,999,"English, French, German, Italian, Spanish - Sp...",Counter-Strike,5199,"10,000,000 .. 20,000,000","{'linux': True, 'mac': True, 'windows': True}",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. ...,"{'1980s': 266.0, '1990's': 1191.0, '2.5D': Non...",game,
1,1000000,1000000,"[Single-player, Partial Controller Support, St...",0,IndigoBlue Game Studio,0,"Action, Adventure, Indie",https://cdn.akamai.steamstatic.com/steam/apps/...,999,"English, Korean, Simplified Chinese",ASCENXION,5,"0 .. 20,000","{'linux': False, 'mac': False, 'windows': True}",27,999,PsychoFlux Entertainment,2021/05/14,0,ASCENXION is a 2D shoot 'em up game where you ...,"{'1980s': None, '1990's': None, '2.5D': None, ...",game,
2,1000010,1000010,"[Single-player, Partial Controller Support, St...",99,NEXT Studios,70,"Adventure, Indie, RPG, Strategy",https://cdn.akamai.steamstatic.com/steam/apps/...,1999,"Simplified Chinese, English, Japanese, Traditi...",Crown Trick,646,"200,000 .. 500,000","{'linux': False, 'mac': False, 'windows': True}",4032,599,"Team17, NEXT Studios",2020/10/16,0,"Enter a labyrinth that moves as you move, wher...","{'1980s': None, '1990's': None, '2.5D': None, ...",game,
3,1000030,1000030,"[Multi-player, Single-player, Co-op, Steam Ach...",76,Vertigo Gaming Inc.,0,"Action, Indie, Simulation, Strategy",https://cdn.akamai.steamstatic.com/steam/apps/...,1999,English,"Cook, Serve, Delicious! 3?!",115,"100,000 .. 200,000","{'linux': False, 'mac': True, 'windows': True}",1575,1999,Vertigo Gaming Inc.,2020/10/14,0,"Cook, serve and manage your food truck as you ...","{'1980s': None, '1990's': None, '2.5D': None, ...",game,http://www.cookservedelicious.com
4,1000040,1000040,[Single-player],0,DoubleC Games,0,"Action, Casual, Indie, Simulation",https://cdn.akamai.steamstatic.com/steam/apps/...,199,Simplified Chinese,细胞战争,1,"0 .. 20,000","{'linux': False, 'mac': False, 'windows': True}",0,199,DoubleC Games,2019/03/30,0,这是一款打击感十足的细胞主题游戏！操作简单但活下去却不简单，“你”作为侵入人体的细菌病毒，通...,"{'1980s': None, '1990's': None, '2.5D': None, ...",game,


In [0]:
df.describe().toPandas()

,summary,id,appid,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,positive,price,publisher,release_date,required_age,short_description,type,website
0,count,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691
1,mean,1025603.0926720655,1025603.0926720655,138.9596164550825,67391.3,2.603777989262179,None,None,797.5663033524268,None,Infinity,241.8376937027527,None,1470.8755992889335,773.2849832109317,2498.75,None,0.1978882344490734,None,None,None
2,stddev,522784.9683283426,522784.9683283426,6002.067909130785,210681.95381245477,12.887080174743156,None,None,1104.7624778413053,None,NaN,5765.4137615598665,None,30982.733479535047,1093.134582723431,1809.1987867561706,None,2.296292461481821,None,None,None
3,min,10,10,0,,0,,https://cdn.akamai.steamstatic.com/steam/apps/...,0,,Fieldrunners 2,0,"0 .. 20,000",0,0,,,0,,game,
4,max,999990,2190950,874053,＼上／,90,Web Publishing,https://cdn.akamai.steamstatic.com/steam/apps/...,9999,Turkish,～Daydream～蝶が舞う頃に,908515,"500,000 .. 1,000,000",5943345,9999,Ｌｅｍｏｎ Ｂａｌｍ,2022/11/7,MA 15+,🚗 Take part in a roller coaster of emotions wi...,hardware,www.windybeard.com


### 1.3 Nettoyage et création des variables _dérivées_

Exploration de la colonne "type".

In [0]:
type_counts_df = df.groupBy("type").count().toPandas()
display(type_counts_df)

type,count
hardware,1
game,55690


Il s'avère que cette colonne contient de modalités : "hardware" qui n'a qu'une occurence et "game" qui concerne toutes les autres lignes.

Nous souhaitons comprendre ce qu'est "hardware"

In [0]:
hardware_df = df.filter(F.col('type') == 'hardware').toPandas()
display(hardware_df)

id,appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website
353380,353380,"List(Full controller support, Remote Play Together)",0,,0,,https://cdn.akamai.steamstatic.com/steam/apps/353380/header.jpg?t=1617990330,0,,Steam Link,1771,"500,000 .. 1,000,000","List(true, true, true)",5803,0,Anima Locus,2015/11/10,0,"Extend your Steam gaming experience to your mobile device, TV, or another PC - all you need is a local network or internet connection. In addition, the Steam Link app now supports Remote Play Together. Now you can join games hosted on a friend’s PC just by clicking a link.","List(null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 10, null, 8, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 11, null, null, null, null, null, null, null, 6, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 15, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 6, null, null, null, null, null, null, null, null, null, 10, null, 16, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 23, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 39, null, null, null, null, null, null, null, null, null, null, null, null, null, 440, null, null, null, null, null, null, null, null, null, 11, 17, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null)",hardware,https://store.steampowered.com/remoteplay


la modalité "hardware" concerne donc le Steam link, un accessoire de streaming sans fil vers un écran déporté. cela n'intéresse pas notre travail dans le cadre de ce projet. nous pouvons supprimer la modalité "hardware".

In [0]:
df = df.filter(F.col('type') != 'hardware')
df_copy=df.groupBy("type").count().toPandas()
display(df_copy)

type,count
game,55690


#### 1.4 Nettoyage des types des colonnes et ajout de variables

In [0]:

#initialprice
df = df.withColumn(
    "initialprice_eur",
    F.col("initialprice").cast("double") / 100
)

#price
df = df.withColumn(
    "price_eur",
    F.col("price").cast("double") / 100
)

#required_age
df = df.withColumn(
    "required_age_num",
    F.regexp_extract(F.col("required_age"), r"\d+", 0).cast("int")
)

#jeux adultes
df = df.withColumn(
    "is_adult_game",
    F.col("required_age_num") >= 18
)

#owners : c'est initialement un intervalle du nombre min et max de personnes possédant le jeu. On commence par nettoyer la colonne puis on calcul un nombre moyen de propriétaires (min+max)/2.
df = (df
    .withColumn("owners_clean", F.regexp_replace(F.col("owners"), r"\s", ""))
    .withColumn("owners_min",   F.regexp_extract(F.col("owners_clean"), r"^([\d,]+)\.\.", 1))
    .withColumn("owners_max",   F.regexp_extract(F.col("owners_clean"), r"\.\.([\d,]+)$", 1))
    .withColumn("owners_min",   F.regexp_replace(F.col("owners_min"), r",", "").cast("long"))
    .withColumn("owners_max",   F.regexp_replace(F.col("owners_max"), r",", "").cast("long"))
    .withColumn("owners_avg",   F.when(F.col("owners_min").isNotNull() & F.col("owners_max").isNotNull(),((F.col("owners_min") + F.col("owners_max"))/2).cast("double")))
)

#release_date
df = (
    df.withColumn(
        "release_date_dt",
        F.coalesce(
            F.try_to_date("release_date", "MMM d, yyyy"),
            F.try_to_date("release_date", "MMM dd, yyyy"),
            F.try_to_date("release_date", "yyyy-MM-dd"),
            F.try_to_date("release_date", "yyyy/M/d"),
            F.try_to_date("release_date", "yyyy/MM/dd"),

            # format yyyy/MM -> on ajoute /01
            F.try_to_date(F.concat(F.col("release_date"), F.lit("/01")), "yyyy/MM/dd")
        )
    )
    .withColumn("release_year", F.year("release_date_dt"))
    .withColumn("release_month", F.month("release_date_dt"))
)
#reviews : ajout d'un ratio d'avis positifs et d'un score
df = (df
    .withColumn("reviews_total", (F.coalesce(F.col("positive"), F.lit(0)) + F.coalesce(F.col("negative"), F.lit(0))).cast("long"))
    .withColumn(
        "pos_ratio",
        F.when(F.col("reviews_total") > 0, (F.col("positive") / F.col("reviews_total")).cast("double"))
    )
    .withColumn(
        "steam_score",
        F.when(F.col("reviews_total") > 0, (F.col("pos_ratio") * 100).cast("double"))
    )
)

# 2. Création des dataframes auxiliaires

### 2.1 table des tags

In [0]:
tags_fields = df.schema["tags"].dataType.names
print(len(tags_fields))
print(tags_fields[:20])

441
['1980s', "1990's", '2.5D', '2D', '2D Fighter', '2D Platformer', '360 Video', '3D', '3D Fighter', '3D Platformer', '3D Vision', '4 Player Local', '4X', '6DOF', '8-bit Music', 'ATV', 'Abstract', 'Action', 'Action RPG', 'Action RTS']


In [0]:
tags_cols = df.schema["tags"].dataType.names

df_map = df.withColumn(
    "tags_map",
    map_from_arrays(
        array(*[lit(c) for c in tags_cols]),
        array(*[col(f"tags.`{c}`") for c in tags_cols])
    )
)

df_long = df_map.select(
    "appid",
    explode("tags_map").alias("tag", "value")
)
df_long = df_long.filter(col("value").isNotNull() & (col("value") > 0))

top_tags = (
    df_long
    .groupBy("tag")
    .agg(count("*").alias("occurrences"))
    .orderBy(col("occurrences").desc())
)

In [0]:
top_tags.show(top_tags.count(),truncate=False)

+---------------------------------+-----------+
|tag                              |occurrences|
+---------------------------------+-----------+
|Indie                            |36062      |
|Singleplayer                     |27344      |
|Action                           |25022      |
|Casual                           |23993      |
|Adventure                        |23150      |
|2D                               |14547      |
|Strategy                         |11849      |
|Simulation                       |11599      |
|RPG                              |10161      |
|Puzzle                           |9719       |
|Atmospheric                      |9232       |
|Early Access                     |7629       |
|Pixel Graphics                   |7521       |
|3D                               |7430       |
|Story Rich                       |7359       |
|Multiplayer                      |6994       |
|Colorful                         |6974       |
|Arcade                           |6516 

On remarque que "tags" contient à la fois des éléments qui décrivent le type de jeu (rpg,MMORPG,Simulation,Shooter,Platformer etc.) mais également, l'univers dans lequel se situe le jeu (Sci-fi,Medieval, etc.) ou encore l'esthétique ou le gameplay. La colonne n'est pas exploitable telle quelle, il y a trop de nettoyage à faire dessus.

==> Nous préférons la supprimer du dataframe principal et lui consacrer un dataframe spécifique pour améliorer les performances de traitement et alléger le schéma.

In [0]:
tag_cols = df.select("tags.*").columns

df_tags = (
    df.select(
        "appid",
        F.explode(
            F.array([
                F.struct(
                    F.lit(c).alias("tag"),
                    F.col(f"tags.`{c}`").alias("occurrences")
                )
                for c in tag_cols
            ])
        ).alias("tag_struct")
    )
    .select(
        "appid",
        F.col("tag_struct.tag"),
        F.col("tag_struct.occurrences")
    )
    .filter(F.col("occurrences") > 0)
)

In [0]:
print("Structure du dataframe df_tags :")
df_tags.printSchema()

Structure du dataframe df_tags :
root
 |-- appid: long (nullable = true)
 |-- tag: string (nullable = false)
 |-- occurrences: long (nullable = true)



Nous allons créer trois autres dataframes pour améliorer les performances et rendre la structure de l'information plus cohérente :

- df_platforms : listant les différentes plateformes sur lesquelles les jeux sont déployées
- df_categories : listant les différentes catégories de classification des jeux
- df_publisher : liste des éditeurs des jeux.

### 2.2 table des plateformes

In [0]:
df_platforms = (
    df.select(
        "appid",
        F.col("platforms.windows").alias("Windows"),
        F.col("platforms.mac").alias("Mac"),
        F.col("platforms.linux").alias("Linux"),
    )
)

print("Structure du dataframe df_platforms :")
df_platforms.printSchema()

Structure du dataframe df_platforms :
root
 |-- appid: long (nullable = true)
 |-- Windows: boolean (nullable = true)
 |-- Mac: boolean (nullable = true)
 |-- Linux: boolean (nullable = true)



### 2.3 table des catégories

In [0]:
df_categories = (
    df
    .select("appid", F.explode("categories").alias("category"))
    
    .withColumn(
        "category_group",
        
        # Gameplay mode
        F.when(F.col("category").isin(
            "Single-player",
            "Multi-player",
            "Online Multi-Player",
            "LAN Multi-Player",
            "Shared/Split Screen",
            "Shared/Split-Screen",
            "Co-op",
            "Online Co-op",
            "LAN Co-op",
            "MMO",
            "PvP",
            "Online PvP",
            "LAN PvP",
            "Shared/Split Screen PvP",
            "Cross-Platform Multiplayer",
            "Shared/Split Screen Co-op",
            "Steam Turn Notifications"
        ), "Gameplay mode")
        
        # Community / Social
        .when(F.col("category").isin(
            "Steam Achievements",
            "Steam Leaderboards",
            "Steam Workshop",
            "Stats",
            "Includes level editor",
            "Mods",
            "Mods (require HL2)",
            "Includes Source SDK",
            "Valve Anti-Cheat Activated",
            "Valve Anti-Cheat enabled"
        ), "Community / Social")
        
        # Device / Hardware support
        .when(F.col("category").isin(
            "Full controller support",
            "Partial Controller Support",
            "Tracked Controller Support",
            "VR Supported",
            "VR Support",
            "VR Only",
            "Steam Cloud",
            "Remote Play on Phone",
            "Remote Play on Tablet",
            "Remote Play on TV",
            "Remote Play Together"
        ), "Device / Hardware support")
        
        # Accessibility / UX
        .when(F.col("category").isin(
            "Captions available",
            "Commentary available"
        ), "Accessibility / UX")
        
        # Monetization / Content
        .when(F.col("category").isin(
            "In-App Purchases",
            "Downloadable Content",
            "Steam Trading Cards",
            "SteamVR Collectibles"
        ), "Monetization / Content")
        
        # fallback
        .otherwise("Other")
    )
)

print("Structure du dataframe df_categories :")
df_categories.printSchema()

Structure du dataframe df_categories :
root
 |-- appid: long (nullable = true)
 |-- category: string (nullable = true)
 |-- category_group: string (nullable = false)



### 2.4 table des éditeurs

In [0]:
suffixes = [
    ", Inc.",
    ", Ltd.",
    ", LTD.",
    ", LLC",
    ", Co., Ltd.",
    ", Co.,Ltd.",
    ", GmbH",
    ", S.A.",
    ", S.r.l."
]

df_publisher = df.withColumn("publisher_clean", F.col("publisher"))

for s in suffixes:
    df_publisher = df_publisher.withColumn(
        "publisher_clean",
        F.regexp_replace("publisher_clean", s, s.replace(",", "§"))
    )

df_publisher = (
    df_publisher
    .withColumn("publisher", F.explode(F.split(F.col("publisher_clean"), ",")))
    .withColumn("publisher", F.trim(F.col("publisher")))
    .withColumn("publisher", F.regexp_replace("publisher", "§", ","))
    .filter(F.col("publisher") != "")
    .select("appid", "publisher")
)

print("Structure du dataframe df_publisher :")
df_publisher.printSchema()

Structure du dataframe df_publisher :
root
 |-- appid: long (nullable = true)
 |-- publisher: string (nullable = false)



### 2.5 Table des genres

In [0]:
df_genres = (
    df
    .select("appid", "genre")
    .filter(F.col("genre").isNotNull())
    .withColumn("genre_array", F.split(F.col("genre"), ","))
    .withColumn("genre", F.explode("genre_array"))
    .withColumn("genre", F.trim(F.col("genre")))
    .filter(F.col("genre") != "")
    .select("appid", "genre")
)
print("Structure du dataframe df_genres :")
df_genres.printSchema()

Structure du dataframe df_genres :
root
 |-- appid: long (nullable = true)
 |-- genre: string (nullable = false)



### 2.6 Suppression des colonnes inutiles dans la table principale

Sélection des colonnes à supprimer du dataframe principal.

In [0]:
col_to_drop =[
    "id",
    "developer",
    "tags",
    "categories",
    "platforms",
    "ccu",
    "publisher",
    "header_image",
    "short_description",
    "website"
]

In [0]:
df = df.drop(*col_to_drop)
print("Structure du dataframe principal df :")
df.printSchema()

Structure du dataframe principal df :
root
 |-- appid: long (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- type: string (nullable = true)
 |-- initialprice_eur: double (nullable = true)
 |-- price_eur: double (nullable = true)
 |-- required_age_num: integer (nullable = true)
 |-- is_adult_game: boolean (nullable = true)
 |-- owners_clean: string (nullable = true)
 |-- owners_min: long (nullable = true)
 |-- owners_max: long (nullable = true)
 |-- owners_avg: double (nullable = true)
 |-- release_date_dt: date (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rele

test de la présence de doublons dans df

In [0]:
df.groupBy("appid").count().orderBy(F.desc("count")).show(10)

+-------+-----+
|  appid|count|
+-------+-----+
|1005130|    1|
|1002420|    1|
|1003090|    1|
|1003940|    1|
|1000990|    1|
|1002360|    1|
|1000470|    1|
|1001960|    1|
|1001040|    1|
|1003750|    1|
+-------+-----+
only showing top 10 rows


In [0]:
df.printSchema()

root
 |-- appid: long (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- type: string (nullable = true)
 |-- initialprice_eur: double (nullable = true)
 |-- price_eur: double (nullable = true)
 |-- required_age_num: integer (nullable = true)
 |-- is_adult_game: boolean (nullable = true)
 |-- owners_clean: string (nullable = true)
 |-- owners_min: long (nullable = true)
 |-- owners_max: long (nullable = true)
 |-- owners_avg: double (nullable = true)
 |-- release_date_dt: date (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- release_month: integer (nullable = true)
 

Contrôle des outliers

recherche des outliers prix

In [0]:
df.orderBy(F.desc("price_eur")).select("name","price_eur").show(20, False)

+-----------------------------------------------------------------+---------+
|name                                                             |price_eur|
+-----------------------------------------------------------------+---------+
|Ascent Free-Roaming VR Experience                                |999.0    |
|Aartform Curvy 3D 3.0                                            |299.9    |
|Houdini Indie                                                    |269.99   |
|VEGAS Edit 20 Steam Edition                                      |249.0    |
|Arcade Drift                                                     |199.99   |
|VR Long March                                                    |199.99   |
|fight                                                            |199.99   |
|眼睛（眼球）结构研究                                             |199.99   |
|Earthquake escape                                                |199.99   |
|VRemedies - MRI Procedure Experience                             |199.99 

Les prix semblent très important, trop pour des jeux. Il s'agit en fait d'outils de développement, d'expériences VR ou, pour "Ascent free roaming VR experience", d'un jeu intégrant la VR, des sols vibrants, les sensations aptiques etc. Ce jeu est principalement destiné aux professionnels ayant un équipement spécifique. 

recherche de ratios négatifs ou >1

In [0]:
df.select(F.min("pos_ratio"),F.max("pos_ratio")).show()

+--------------+--------------+
|min(pos_ratio)|max(pos_ratio)|
+--------------+--------------+
|           0.0|           1.0|
+--------------+--------------+



Il ne semble pas y avoir d'éléments problématiques.

# 3. Croissance du catalogue Steam

### 3.1 Nombre de jeux publiés chaque année

In [0]:
df_games_per_year = (
    df.filter(F.col("release_year").isNotNull())
      .groupBy("release_year")
      .agg(F.count("*").alias("n_games"))
      .orderBy("release_year")
)
display(df_games_per_year)

release_year,n_games
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


Databricks visualization. Run in Databricks to view.

Les volumes de sorties annuelles augmentent fortement entre 2012 et 2018, ce qui correspond à une phase d’expansion rapide du catalogue Steam.

À partir de 2019, la croissance ralentit et le nombre de nouvelles publications semble se stabiliser, voire légèrement diminuer certaines années.

Ce ralentissement apparaît avant la pandémie de COVID-19. Il est donc difficile de l’attribuer directement à cet événement. L’année 2020, la plus fortement marquée par la pandémie, ne montre pas de chute brutale du nombre de sorties et s’inscrit plutôt dans une phase de reprise du volume de publications.

On peut formuler l’hypothèse que les périodes de confinement ont pu favoriser le développement de projets indépendants, mais cette interprétation ne peut pas être confirmée à partir des données disponibles dans ce dataset.

#### 3.2 Nombre d'éditeurs ayant sorti au moins un jeu chaque année

In [0]:
df_publishers_per_year = (
    df.filter(F.col("release_year").isNotNull())
      .select("appid", "release_year")
      .join(df_publisher.select("appid", "publisher"), on="appid", how="inner")
      .filter(F.col("publisher").isNotNull() & (F.trim(F.col("publisher")) != ""))
      .groupBy("release_year")
      .agg(F.countDistinct("publisher").alias("n_publishers"))
      .orderBy(F.col("release_year").asc())
)

display(df_publishers_per_year)

release_year,n_publishers
1997,2
1998,1
1999,2
2000,1
2001,3
2002,1
2003,3
2004,2
2005,6
2006,30


Databricks visualization. Run in Databricks to view.

l'évolution du nombre d'éditeurs suit exactement la même tendance que pour le nombre de jeux sortis par année, seuls les volumes varient : ils sont logiquement moins importants pour le nombre d'éditeurs.

# 4. Structure des éditeurs

### 4.1 Top éditeurs par nombre de jeux

In [0]:

df_games_per_publisher = (
    df_publisher
    .groupBy("publisher")
    .agg(F.countDistinct("appid").alias("n_games"))
    .orderBy(F.desc("n_games"))
    .limit(20)
)

display(df_games_per_publisher)

publisher,n_games
Big Fish Games,423
8floor,202
SEGA,187
Square Enix,151
Strategy First,151
THQ Nordic,144
Choice of Games,140
Sekai Project,136
Plug In Digital,136
HH-Games,132


Databricks visualization. Run in Databricks to view.

On remarque que ce sont majoritairement des éditeurs indépendants qui sortent le plus de jeux. Ceci semble logique dans le sens où les jeux indépendant sont souvent moins lourds à développer que des AAA de grosses franchises. cette répartition est cohérente avec la structure du catalogue de jeux Steam.

Quelques très gros éditeurs apparaissent cependant dans le top 20 : Sega, Square Enix, THQ Nordic, Electronic Arts (EA), Ubisoft

### 4.2 Top 15 du nombre de jeux sortis par éditeur par année

In [0]:
TOP_N = 15

# Base: année par appid
df_year = (
    df.select("appid", "release_year")
      .filter((F.col("release_year") >= 2012) & (F.col("release_year") <= 2022))
      .filter(F.col("release_year").isNotNull())
)

# Publishers: 1 ligne par (appid, publisher) déjà nettoyé chez toi
df_pub = (
    df_publisher
      .select("appid", F.col("publisher").alias("publisher_clean"))
      .filter(F.col("publisher_clean").isNotNull() & (F.trim(F.col("publisher_clean")) != ""))
)

# Join + agrégat
df_pub_year = (
    df_year.join(df_pub, on="appid", how="inner")
           .groupBy("release_year", "publisher_clean")
           .agg(F.count("*").alias("n_games"))
)

# Top N par année
w = Window.partitionBy("release_year").orderBy(F.desc("n_games"), F.asc("publisher_clean"))

df_pub_top = (
    df_pub_year
      .withColumn("rank", F.row_number().over(w))
      .filter(F.col("rank") <= TOP_N)
      .drop("rank")
)

# Collect pour Plotly
pdf = df_pub_top.orderBy(F.desc("release_year"), F.desc("n_games")).toPandas()
years = sorted(pdf["release_year"].unique(), reverse=True)

# Subplots
ncols = 3
nrows = math.ceil(len(years) / ncols)

fig = make_subplots(
    rows=nrows,
    cols=ncols,
    subplot_titles=[str(y) for y in years],
    horizontal_spacing=0.08,
    vertical_spacing=0.12
)

for i, year in enumerate(years):
    r = i // ncols + 1
    c = i % ncols + 1

    d = (
        pdf[pdf["release_year"] == year]
        .sort_values(["n_games", "publisher_clean"], ascending=[False, True])
        .head(TOP_N)
    )

    fig.add_trace(
        go.Bar(
            x=d["n_games"],
            y=d["publisher_clean"],
            orientation="h",
            showlegend=False
        ),
        row=r,
        col=c
    )

    fig.update_yaxes(autorange="reversed", row=r, col=c)

fig.update_layout(
    height=320 * nrows,
    title=f"Top {TOP_N} publishers par année (2012–2022)",
    margin=dict(l=40, r=20, t=60, b=30)
)

fig.show()

On remarque que, pour les années 2012 et 2013, des très gros éditeurs étaient présents dans le top des volumes de sorties (2012 : SEGA, THQ Nordic ; 2013 : Square Enix, Ubisoft, BANDAI NAMCO, Paradox Interactive).

cependant, les années suivantes, ils disparaissent globalement du top 15. Cela peut s'expliquer par l'accroissement du volume de jeux sortis par les développeurs les plus prolifiques. En 2012 et 2013, l'éditeur ayant sorti le plus grand nombre de jeux en a sorti, respectivement 21 et 11.... en 2021 et 2022, ce sont 66 et 46 jeux qui ont été sortis en une année par un éditeur...

le pic s'étant produit en 2017 avec 112 jeux pour Big Fish Games !!!

Certains éditeurs sont spécialisés dans des genres de petits jeux courts et peu coûteux à développer  et peuvent, ainsi, en sortir une grande quantité en une année.

# 5. Plateformes

### 5.1 Ventilation des jeux par plateforme supportée

In [0]:
total_games = df_platforms.select(F.countDistinct("appid")).collect()[0][0]

df_platform_share = (
    df_platforms
    .agg(
        F.sum(F.col("Windows").cast("int")).alias("windows_games"),
        F.sum(F.col("Mac").cast("int")).alias("mac_games"),
        F.sum(F.col("Linux").cast("int")).alias("linux_games")
    )
    .withColumn("windows_pct", F.round(F.col("windows_games")/total_games*100,2))
    .withColumn("mac_pct", F.round(F.col("mac_games")/total_games*100,2))
    .withColumn("linux_pct", F.round(F.col("linux_games")/total_games*100,2))
)

display(df_platform_share)

windows_games,mac_games,linux_games,windows_pct,mac_pct,linux_pct
55675,12769,8457,99.97,22.93,15.19


Databricks visualization. Run in Databricks to view.

La totalité des jeux présents dans le catalogue Steam au moment de l'extraction est jouable sur PC. Ce n'est pas le cas du tout pour les Mac (22,93% des jeux sont jouables sur Mac) et encore moins pour Linux (15,19% des jeux sont jouables sur Linux).

### 5.2 part des jeux "Windows only" dans l'extraction

In [0]:
df_platform_types = (
    df_platforms
    .withColumn(
        "platform_type",
        F.when(
            (F.col("Windows") == True) &
            (F.col("Mac") == False) &
            (F.col("Linux") == False),
            "Windows only"
        )
        .when(
            (F.col("Windows") == True) &
            ((F.col("Mac") == True) | (F.col("Linux") == True)),
            "Windows + Mac/Linux"
        )
        .otherwise("Mac/Linux without Windows")
    )
)

total_games = df_platform_types.select(F.countDistinct("appid")).collect()[0][0]

df_platform_distribution = (
    df_platform_types
    .groupBy("platform_type")
    .agg(F.countDistinct("appid").alias("n_games"))
    .withColumn(
        "share_pct",
        F.round(F.col("n_games") / F.lit(total_games) * 100, 2)
    )
)

display(df_platform_distribution)

platform_type,n_games,share_pct
Mac/Linux without Windows,15,0.03
Windows + Mac/Linux,14404,25.86
Windows only,41271,74.11


Databricks visualization. Run in Databricks to view.


Une grande majorité des jeux sont édités "Windows only", ce qui signifie qu'ils ne seront jamais adaptés pour d'autres plateformes. Cela représente, au moment de l'extraction du dataset, 74,11% du catalogue disponible.

Mac et Linux sont relégués à un rôle de support secondaire. Cette part très importante de jeux Windows only biaise la distribution globale.

#### 5.2.1 Evolution de la part des jeux "Windows only" dans le temps

In [0]:
df_platform_year = (
    df_platforms
    .join(df.select("appid","release_year"), "appid")
    .filter(F.col("release_year").isNotNull())
    .filter(F.col("release_year") > 2006)
)

df_platform_year = df_platform_year.withColumn(
    "windows_only",
    (
        (F.col("Windows") == True) &
        (F.col("Mac") == False) &
        (F.col("Linux") == False)
    ).cast("int")
)

df_windows_only_share = (
    df_platform_year
    .groupBy("release_year")
    .agg(
        F.countDistinct("appid").alias("n_games"),
        F.sum("windows_only").alias("windows_only_games")
    )
    .withColumn(
        "windows_only_pct",
        F.round(F.col("windows_only_games") / F.col("n_games") * 100, 2)
    )
    .orderBy("release_year")
)

display(
    df_windows_only_share.select(
        "release_year",
        "windows_only_pct"
    )
)

release_year,windows_only_pct
2007,88.78
2008,88.05
2009,78.14
2010,65.63
2011,67.04
2012,57.39
2013,50.32
2014,55.49
2015,54.49
2016,62.72


Databricks visualization. Run in Databricks to view.


On remarque qu'entre 2008 et 2013, la part de jeux sortis chaque année et réservés à Windows a fortement diminué, passant de 88,78% en 2007 à 50,32% en 2013.

A partir de 2014, avec l'explosion des publications de jeux sur Steam, la part des jeux "Windows only" a de nouveau cru et atteint, pour les jeux sortis en 2022 82,55% soit légèrement moins qu'en 2007.

### 5.3 Evolution de la part des jeux disponible pour chaque plateforme selon l'année de sortie

In [0]:

df_platform_year = (
    df_platforms
    .join(df.select("appid", "release_year"), "appid")
    .filter(F.col("release_year").isNotNull())
    .filter(F.col("release_year")>=2006)
)

df_platform_year_share = (
    df_platform_year
    .groupBy("release_year")
    .agg(
        F.countDistinct("appid").alias("n_games"),
        F.sum(F.col("Windows").cast("int")).alias("windows_games"),
        F.sum(F.col("Mac").cast("int")).alias("mac_games"),
        F.sum(F.col("Linux").cast("int")).alias("linux_games")
    )
    .withColumn("windows_pct", F.round(F.col("windows_games")/F.col("n_games")*100,2))
    .withColumn("mac_pct", F.round(F.col("mac_games")/F.col("n_games")*100,2))
    .withColumn("linux_pct", F.round(F.col("linux_games")/F.col("n_games")*100,2))
    .orderBy("release_year")
)

display(
    df_platform_year_share.select(
        "release_year",
        "windows_pct",
        "mac_pct",
        "linux_pct"
    )
)

release_year,windows_pct,mac_pct,linux_pct
2006,100.0,26.23,16.39
2007,100.0,11.22,4.08
2008,100.0,10.06,3.77
2009,100.0,21.22,8.68
2010,100.0,34.03,20.49
2011,100.0,30.71,17.98
2012,99.71,41.74,27.25
2013,100.0,46.71,38.22
2014,100.0,42.45,29.67
2015,99.96,41.94,30.64


Databricks visualization. Run in Databricks to view.

Au début des années 2010, grâce à une visibilité croissante des joueurs Linux et Mac demandant un accès à une bibliothèque plus importante, un passage des nouveaux Mac sur des puces Intel (entre 2006 et 2007) facilitant la portabilité des jeux sur cette plateforme, l'arrivée d'Android (basé sur Linux) ainsi qu'une amélioration en termes de qualité et de fréquence de mise à jour des pilotes de cartes graphiques pour Linux par les deux géants AMD et Nvidia, la proportion de jeux accessibles aux ordinateurs Mac et Linux a crû pendant quelques années pour atteindre un pic à 46,71% des jeux sortis en 2013 disponibles sur Mac et 38,22% pour Linux. Cette tendance s'est inversée à partir de 2014 pour redescendre doucement, pour les sorties 2022, à un niveau égale à celui de 2007 pour les Mac et 2009 pour Linux.

#### 5.3.1 Evolution de la part des jeux disponibles sur chaque plateforme, en excluant les jeux "Windows Only"

In [0]:
df_platform_year = (
    df_platforms
    .join(df.select("appid", "release_year"), "appid")
    .filter(F.col("release_year").isNotNull())
    .filter(F.col("release_year") >= 2006)

    # exclusion des jeux Windows-only
    .filter(
        ~(
            (F.col("Windows") == True) &
            (F.col("Mac") == False) &
            (F.col("Linux") == False)
        )
    )
)

df_platform_year_share = (
    df_platform_year
    .groupBy("release_year")
    .agg(
        F.countDistinct("appid").alias("n_games"),
        F.sum(F.col("Windows").cast("int")).alias("windows_games"),
        F.sum(F.col("Mac").cast("int")).alias("mac_games"),
        F.sum(F.col("Linux").cast("int")).alias("linux_games")
    )
    .withColumn("windows_pct", F.round(F.col("windows_games")/F.col("n_games")*100,2))
    .withColumn("mac_pct", F.round(F.col("mac_games")/F.col("n_games")*100,2))
    .withColumn("linux_pct", F.round(F.col("linux_games")/F.col("n_games")*100,2))
    .orderBy("release_year")
)

display(
    df_platform_year_share.select(
        "release_year",
        "windows_pct",
        "mac_pct",
        "linux_pct"
    )
)

release_year,windows_pct,mac_pct,linux_pct
2006,100.0,100.0,62.5
2007,100.0,100.0,36.36
2008,100.0,84.21,31.58
2009,100.0,97.06,39.71
2010,100.0,98.99,59.6
2011,100.0,93.18,54.55
2012,99.32,97.96,63.95
2013,100.0,94.02,76.92
2014,100.0,95.38,66.67
2015,99.91,92.15,67.32


Databricks visualization. Run in Databricks to view.

Il est nécessaire de garder en tête que ce graphique ne représente que 26% du catalogue global Steam au moment de l'extraction.

Cependant, on observe une nette progression des proportions de jeux publiés à la fois sur Windows et sur au moins l'une des autres plateformes.

Jusqu'en 2012, la quasi totalité des jeux (hors "Windows Only") était publiée également sur Mac. Cette proportioncommence à diminuer de manière régulière à partir de 2013 et continue jusqu'en 2022 pour atteindre 80,1%.

Concernant Linux, après une forte progression sur la période 2008 - 2013 où 76,9% des jeux (hors "Windows Only") ont également été publiés sur cette plateforme, on constate une diminution jusqu'en 2019. Cette baisse correspond à la période où la publication de jeux a explosé, sur Steam.

Entre 2019 et 2022, avec la baisse du nombre de jeux sortis chaque année, la part des jeux publiés également sur Linux a, de nouveau, recommencé à croitre.

In [0]:

df_platform_year = (
    df_platforms
    .join(df.select("appid", "release_year"), "appid")
    .filter(F.col("release_year").isNotNull())
    .filter(F.col("release_year") >= 2006)
)

df_platform_counts = (
    df_platform_year
    .groupBy("release_year")
    .agg(
        F.countDistinct("appid").alias("n_games_total"),
        F.countDistinct(
            F.when(F.col("Mac") == True, F.col("appid"))
        ).alias("mac_games"),
        F.countDistinct(
            F.when(F.col("Linux") == True, F.col("appid"))
        ).alias("linux_games")
    )
    .orderBy("release_year")
)

display(
    df_platform_counts.select(
        "release_year",
        "mac_games",
        "linux_games"
    )
)

release_year,mac_games,linux_games
2006,16,10
2007,11,4
2008,16,6
2009,66,27
2010,98,59
2011,82,48
2012,144,94
2013,220,180
2014,661,462
2015,1080,789


Databricks visualization. Run in Databricks to view.

Cette baisse proportionnelle (période 2014-2019 pour Linux et 2013-2022 pour Mac) est, en partie, un trompe l'oeil car le volume de jeux édités pour Mac et Linux est en croissance constante jusqu'en 2018 mais le volume total de jeux édités chaque année a crû encore plus rapidement et écrase donc la propotion de jeux disponibles sur Mac et Linux.

A partir de 2019 on constate, comme pour Windows, une diminuton du volume de sorties.

### 5.4 Genres les plus représentés par plateforme

In [0]:
# Passage des plateformes en format long
df_platform_long = (
    df_platforms
    .select("appid", "Windows", "Mac", "Linux")
    .withColumn("platform_array",
        F.array(
            F.struct(F.lit("Windows").alias("platform"), F.col("Windows").alias("available")),
            F.struct(F.lit("Mac").alias("platform"), F.col("Mac").alias("available")),
            F.struct(F.lit("Linux").alias("platform"), F.col("Linux").alias("available"))
        )
    )
    .withColumn("platform_struct", F.explode("platform_array"))
    .select(
        "appid",
        F.col("platform_struct.platform").alias("platform"),
        F.col("platform_struct.available").alias("available")
    )
    .filter(F.col("available") == True)
)

# Jointure genres + plateformes
df_genre_platform = df_genres.join(df_platform_long, "appid")

# Comptage des genres par plateforme
df_genre_distribution = (
    df_genre_platform
    .groupBy("platform", "genre")
    .count()
)

# Calcul des proportions par plateforme
w_platform = Window.partitionBy("platform")

df_genre_distribution = (
    df_genre_distribution
    .withColumn("total_platform", F.sum("count").over(w_platform))
    .withColumn("proportion", (F.col("count") / F.col("total_platform")) * 100)
)

# Top 10 genres par plateforme
w_rank = Window.partitionBy("platform").orderBy(F.desc("proportion"))

top_genres = (
    df_genre_distribution
    .withColumn("rank", F.row_number().over(w_rank))
    .filter(F.col("rank") <= 10)
    .select("platform", "genre", "proportion", "count")
)

display(top_genres)

platform,genre,proportion,count
Linux,Indie,29.08833215223644,6978
Linux,Action,14.085622577014464,3379
Linux,Casual,13.777147859435573,3305
Linux,Adventure,13.764642127641835,3302
Linux,Strategy,7.611822085122348,1826
Linux,Simulation,6.386260369335946,1532
Linux,RPG,6.352911751219309,1524
Linux,Early Access,2.6345408312143066,632
Linux,Free to Play,1.97590562341073,474
Linux,Racing,1.2672474884321983,304


Databricks visualization. Run in Databricks to view.

# 6. Langues

### 6.1 Langues les plus supportées

In [0]:
# Total de jeux uniques dans le dataset (aucune valeur hardcodée)
total_games_df = df.select("appid").distinct().agg(
    F.count("*").alias("total_games")
)

# Explose les langues, nettoie et déduplique au niveau jeu-langue
df_languages_clean = (
    df
    .select("appid", "languages")
    .filter(F.col("appid").isNotNull() & F.col("languages").isNotNull())
    .withColumn("language", F.explode(F.split(F.col("languages"), ",")))
    .withColumn("language", F.trim(F.col("language")))
    .filter(F.col("language") != "")
    .select("appid", "language")
    .dropDuplicates(["appid", "language"])
)

# Nombre de jeux par langue
df_lang_counts = (
    df_languages_clean
    .groupBy("language")
    .agg(F.countDistinct("appid").alias("games_supporting_language"))
)

# Pourcentage de jeux supportant chaque langue
df_lang_pct = (
    df_lang_counts
    .crossJoin(total_games_df)
    .withColumn(
        "pct_games",
        F.round(F.col("games_supporting_language") / F.col("total_games") * 100, 2)
    )
    .orderBy(F.desc("pct_games"), F.desc("games_supporting_language"))
    .limit(20)
)

display(df_lang_pct)

language,games_supporting_language,total_games,pct_games
English,55116,55690,98.97
German,14019,55690,25.17
French,13426,55690,24.11
Russian,12922,55690,23.2
Simplified Chinese,12782,55690,22.95
Spanish - Spain,12233,55690,21.97
Japanese,10368,55690,18.62
Italian,9304,55690,16.71
Portuguese - Brazil,6750,55690,12.12
Korean,6600,55690,11.85


Databricks visualization. Run in Databricks to view.

La totalité ou presque des jeux supporte l'anglais. Les langues les plus supportées, après l'anglais, sont très loin derrière.

Seul 25,17% des jeux supportent l'allemand et 24,11% le français. viennent ensuite le russe (23,2%), le chinois simplifié (22,95%), l'espagnol (21,97)

#### 6.1.1 Jeux ne supportant pas l'anglais

In [0]:
df_languages = (
    df
    .select("appid", "name", "languages")
    .filter(F.col("languages").isNotNull())
    .withColumn("language", F.explode(F.split("languages", ",")))
    .withColumn("language", F.trim("language"))
    .select("appid", "name", "language")
)

df_games_with_english = (
    df_languages
    .filter(F.lower(F.col("language")) == "english")
    .select("appid")
    .distinct()
)

df_games_no_english = (
    df
    .join(df_games_with_english, on="appid", how="left_anti")
)

df_no_eng_languages = (
    df_games_no_english
    .select("appid", "languages")
    .withColumn("language", F.explode(F.split("languages", ",")))
    .withColumn("language", F.trim("language"))
)

display(df_no_eng_languages.groupBy("language") \
    .count() \
    .orderBy(F.desc("count")))


language,count
Simplified Chinese,378
Japanese,96
Traditional Chinese,76
Russian,31
French,13
Korean,13
German,12
,10
Italian,8
Spanish - Spain,5


Databricks visualization. Run in Databricks to view.

Une très faible quantité de jeux ne supporte pas la langue anglaise. Après exploration, il s'avère que ce sont, dans la très grande majorité, des jeux chinois ou japonais.

### 6.2 Evolution du support des langues

In [0]:
# 1) explode + nettoyage
df_languages = (
    df.filter(F.col("release_year").isNotNull()& (F.col("release_year") >= 2006))
      .select("appid", "release_year", "languages")
      .withColumn("language", F.explode(F.split(F.coalesce(F.col("languages"), F.lit("")), ",")))
      .withColumn("language", F.trim(F.col("language")))
      .filter(F.col("language") != "")
      # retire "(...)" et "*"
      .withColumn("language", F.regexp_replace(F.col("language"), r"\(.*?\)", ""))
      .withColumn("language", F.regexp_replace(F.col("language"), r"\*", ""))
      .withColumn("language", F.trim(F.col("language")))
      .filter(F.col("language") != "")
      .select("appid", "release_year", "language")
      .dropDuplicates(["appid", "release_year", "language"])  # évite doubles comptages
)

# 2) top 10 langues globales
top_languages = (
    df_languages.groupBy("language")
    .agg(F.countDistinct("appid").alias("n_games"))
    .orderBy(F.desc("n_games"))
    .limit(10)
    .select("language")
)

df_languages_top = df_languages.join(top_languages, on="language", how="inner")

# 3) numérateur: nb jeux de l'année qui proposent cette langue
df_lang_year = (
    df_languages_top
    .groupBy("release_year", "language")
    .agg(F.countDistinct("appid").alias("n_games_lang"))
)

# 4) dénominateur: nb jeux sortis cette année (tous jeux)
df_year_total = (
    df.filter(F.col("release_year").isNotNull())
      .groupBy("release_year")
      .agg(F.countDistinct("appid").alias("n_games_year"))
)

# 5) part (%) = n_games_lang / n_games_year
df_lang_share = (
    df_lang_year.join(df_year_total, on="release_year", how="inner")
    .withColumn("share_pct", (F.col("n_games_lang") / F.col("n_games_year") * 100.0))
    .orderBy("release_year", F.desc("share_pct"))
)

display(df_lang_share)

release_year,language,n_games_lang,n_games_year,share_pct
2006,English,61,61,100.0
2006,French,25,61,40.98360655737705
2006,German,23,61,37.704918032786885
2006,Italian,21,61,34.42622950819672
2006,Spanish - Spain,19,61,31.147540983606557
2006,Russian,13,61,21.311475409836063
2006,Japanese,6,61,9.836065573770492
2006,Korean,4,61,6.557377049180328
2006,Simplified Chinese,4,61,6.557377049180328
2006,Portuguese - Brazil,1,61,1.639344262295082


Databricks visualization. Run in Databricks to view.

- La quasi totalité des jeux propose l'anglais sur toute la période observée
- On note que la proportion de jeux proposant l'espagnol, le français, l'allemand ou l'italien a diminuée progressivement à partir de 2015 pour se stabiliser à partir de 2017. Ces proportions ont été divisées par 2 pour ces 4 langues.
- A part en 2013 où il y a une proportion étonnament importante de jeux proposant le russe comme langue, la part des jeux qui proposent cette langue reste relativement stable sur toute la période observée.
- A partir de 2012, le portugais (Brésil) émerge significativement mais se stabilise rapidement autour de 12% par an.
- A partir de 2013, émergence du chinois simplifié et progression régulière chaque année jusqu'en 2022.

# 7. Catégories

### 7.1 Grandes classes de catégories

In [0]:
total_games_categories = (
    df_categories
    .select("appid")
    .distinct()
    .count()
)

category_group_distribution = (
    df_categories
    .groupBy("category_group")
    .agg(F.countDistinct("appid").alias("nb_games"))
    .withColumn("proportion_games", F.col("nb_games") / F.lit(total_games_categories) * 100)
    .orderBy(F.desc("proportion_games"))
)

display(category_group_distribution)

category_group,nb_games,proportion_games
Gameplay mode,54421,99.45358187134504
Community / Social,28920,52.85087719298246
Device / Hardware support,27243,49.786184210526315
Monetization / Content,10253,18.737207602339183
Accessibility / UX,1178,2.1527777777777777


Databricks visualization. Run in Databricks to view.

On retrouve, pour la quasi totalité des jeux, une information sur le type de gameplay ("single player", "multi-player", "co-op").

Pour 52,9% des jeux, on retrouve ce que nous avons appelé une catégorie communautaire/sociale : existance d'achievements, de classsement des joueurs, un workshop pour les moddeurs.

Viennent ensuite les catégories liées aux outils supportés ou proposés par le jeu : "Steam Cloud", 'Controller Support", "VR Support" etc. pour lesquels une information est données pour 49,8% des jeux.

Enfin, 18,7% des jeux ont une catégorie affichée concernant un élément de monétisation tel que "trading cards", "VR Collectibles" ou "In-Game Purchases"

et seulement 2% des jeux ont une information sur l'accessibilité "Commentary available" ou "Captions available".

### 7.2 Catégories par classes analytiques

In [0]:
total_games_categories = (
    df_categories
    .select("appid")
    .distinct()
    .count()
)

category_distribution_grouped = (
    df_categories
    .groupBy("category_group", "category")
    .agg(F.countDistinct("appid").alias("nb_games"))
    .withColumn(
        "proportion_games",
        F.col("nb_games") / F.lit(total_games_categories) * 100
    )
)

# classement à l'intérieur de chaque classe analytique
w = Window.partitionBy("category_group").orderBy(F.desc("proportion_games"))

category_distribution_ranked = (
    category_distribution_grouped
    .withColumn("rank_in_group", F.row_number().over(w))
    .orderBy("category_group", "rank_in_group")
)

display(category_distribution_ranked)

category_group,category,nb_games,proportion_games,rank_in_group
Accessibility / UX,Captions available,1038,1.8969298245614035,1
Accessibility / UX,Commentary available,204,0.37280701754385964,2
Community / Social,Steam Achievements,27394,50.062134502923975,1
Community / Social,Steam Leaderboards,5509,10.067616959064328,2
Community / Social,Stats,2926,5.347222222222222,3
Community / Social,Includes level editor,1657,3.0281432748538015,4
Community / Social,Steam Workshop,1608,2.93859649122807,5
Community / Social,Valve Anti-Cheat enabled,111,0.20285087719298245,6
Community / Social,Includes Source SDK,44,0.08040935672514621,7
Community / Social,Mods,2,0.0036549707602339184,8


Databricks visualization. Run in Databricks to view.

Lorsqu'on observe la ventilation des catégories à l'intérieur des classes analytiques que nous avons créées :

- Community/Social : Les "Steam Achievements" sont renseignés dans 50% des jeux et les "Steam Leaderboards" dans 10%
- Device/Hardware support : "Steam Cloud" est rensiegné pour 26% des jeux suivi de "Full Controller Support" (21,7%)
- Gameplay Mode : 95% des jeux proposent un mode "Single Player" et 21% un mode "Multi-player". pour 13% des jeux, l'information est données de l'existence d'un "PvP" ("joueurs contre joueurs") et pour 10%, de la présence d'un mode "Co-op" ("Coopération).
- Monetization / Content : Pour 16,8% des jeux, il est indiqué l'exxistence de "Steam trading cards"

In [0]:
df_categories_perf = (
    df_categories
    .join(
        df.select(
            "appid",
            "owners_avg",
            "reviews_total",
            "pos_ratio",
            "steam_score",
            "price_eur"
        ),
        "appid"
    )
)

In [0]:
popular_games = (
    df
    .filter(
        (F.col("reviews_total") >= 20000) &
        (F.col("steam_score") >= 95)
    )
    .select("appid")
    .distinct()
)

### 7.3 Distribution des catégories Steam parmi les jeux populaires

In [0]:
popular_categories = (
    df_categories
    .join(popular_games, on="appid", how="inner")
    .groupBy("category_group", "category")
    .agg(F.countDistinct("appid").alias("popular_games_count"))
    .withColumn(
        "share_of_popular_games",
        F.col("popular_games_count") / F.lit(popular_games.count()) * 100
    )
    .orderBy(F.desc("share_of_popular_games"))
)

display(popular_categories)

category_group,category,popular_games_count,share_of_popular_games
Gameplay mode,Single-player,145,95.39473684210526
Community / Social,Steam Achievements,117,76.97368421052632
Device / Hardware support,Steam Cloud,111,73.02631578947368
Monetization / Content,Steam Trading Cards,90,59.210526315789465
Device / Hardware support,Full controller support,84,55.26315789473685
Device / Hardware support,Remote Play on Tablet,81,53.289473684210535
Device / Hardware support,Remote Play on Phone,64,42.10526315789473
Device / Hardware support,Remote Play on TV,56,36.84210526315789
Gameplay mode,Multi-player,55,36.18421052631579
Gameplay mode,Co-op,38,25.0


Databricks visualization. Run in Databricks to view.

Les catégories les plus souvent présentes parmis les jeux les plus populaires (ayant plus de 20 000 reviews et un Steam_Score >= 95) sont :

- Single-player
- Steam Achievements
- Steam Cloud

puis, dans un second temps, nous trouvons :

- Steam trading cards
- Full controller support
- Remote play on tablet

# 8. Genres et composition du catalogue

### 8.1 Genres les plus représentés

#### 8.1.1 Fréquences brutes d'apparition des genres dans le catalogue Steam

In [0]:
# Fréquence brute des genres
top_genres_frequency = (
    df_genres
    .groupBy("genre")
    .agg(F.countDistinct("appid").alias("nb_games"))
    .orderBy(F.desc("nb_games"))
    .limit(12)
)

display(top_genres_frequency)

genre,nb_games
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


Databricks visualization. Run in Databricks to view.

#### 8.1.2 Proportion des jeux auxquels sont rattachés les genres dans le catalogue Steam

In [0]:
# Nombre total de jeux ayant au moins un genre renseigné
total_games = (
    df
    .filter(F.col("genre").isNotNull())
    .select("appid")
    .distinct()
    .count()
)

# Distribution globale des genres
df_genre_distribution = (
    df_genres
    .groupBy("genre")
    .agg(F.countDistinct("appid").alias("nb_games"))
    .withColumn("proportion", F.col("nb_games") / F.lit(total_games) * 100)
    .orderBy(F.desc("proportion"))
)

display(df_genre_distribution.limit(12))

genre,nb_games,proportion
Indie,39681,71.25336685221764
Action,23759,42.66295564733345
Casual,22086,39.658825641946486
Adventure,21431,38.48267193391992
Strategy,10895,19.563655952594722
Simulation,10836,19.45771233614653
RPG,9534,17.119770156221943
Early Access,6145,11.034297001256958
Free to Play,3393,6.09265577302927
Sports,2666,4.7872149398455734


Databricks visualization. Run in Databricks to view.

Au moment de l'extraction on retrouve "Indie" (71,25% des jeux ont ce genre attaché) en tant que genre prédominant sur le catalogue Steam. viennent ensuite les genres "Action" (42,66%), Casual" (39,66%) et "Adventure" (38,48%).

il est important de rappeler que les genres ne sont pas exclusifs : un même jeu peut être associé à plusieurs genres. Les proportions observées ne décrivent donc pas une partition du catalogue mais plutôt la fréquence d’apparition de chaque genre parmi les jeux disponibles. Cette caractéristique limite l’interprétation des fréquences individuelles et invite à analyser plus précisément la manière dont les genres se combinent entre eux, ce qui sera exploré dans le second notebook.

### 8.2 Top 10 des genres les plus représentés dans les sorties par année

In [0]:

df_genres_year = (
    df
    .filter((F.col("release_year") >= 2012) & (F.col("release_year") <= 2022))
    .withColumn("genre", F.explode(F.split(F.col("genre"), ",")))
    .withColumn("genre", F.trim(F.col("genre")))
    .filter(F.col("genre") != "")
)


df_counts = (
    df_genres_year
    .groupBy("release_year", "genre")
    .count()
)


w = Window.partitionBy("release_year").orderBy(F.desc("count"), F.asc("genre"))

df_top10 = (
    df_counts
    .withColumn("rk", F.row_number().over(w))
    .filter(F.col("rk") <= 10)
    .drop("rk")
)

pdf = (
    df_top10
    .orderBy(F.desc("release_year"), F.desc("count"))
    .toPandas()
)

years = sorted(pdf["release_year"].unique(), reverse=True)

# Grille (ex: 3 colonnes)
ncols = 3
nrows = math.ceil(len(years) / ncols)

fig = make_subplots(
    rows=nrows, cols=ncols,
    subplot_titles=[str(y) for y in years],
    horizontal_spacing=0.08, vertical_spacing=0.12
)

for i, y in enumerate(years):
    r = i // ncols + 1
    c = i % ncols + 1

    d = pdf[pdf["release_year"] == y].sort_values("count", ascending=False)

    # Pour un bar chart lisible, le plus simple est horizontal
    fig.add_trace(
        go.Bar(
            x=d["count"],
            y=d["genre"],
            orientation="h",
            name=str(y),
            showlegend=False
        ),
        row=r, col=c
    )

    # Top en haut (barh -> inverser l’axe Y)
    fig.update_yaxes(autorange="reversed", row=r, col=c)

fig.update_layout(
    height=300*nrows,
    title_text="Top 10 genres par année (2012–2022)",
    margin=dict(l=40, r=20, t=60, b=30)
)

fig.show()

Sans surprise, le genre "Indie" est le plus représenté chaque année depuis 2013, de très loin.

dans le top 4, on retrouve également les mêmes genres chaque année : "casual", "Action" et "Adventure" qui sont souvent liés au genre "Indie" pour un même jeu.

On remarque que, depuis 2015, le genre "Early Access" est apparu dans ce top 10. Il s'agit en fait de jeu en cours de développement et vendus à un pris généralement plus bas que le prix final afin de financer la poursuite du développement en autorisant les joueurs à profiter du jeu tel qu'il est au cours du développement. C'est devenu une pratique courante de nos jours.

Les jeux "Free to Play" sont présents chaque année avec un volume régulier de sorties. Il s'agit de jeux gratuits dans lesquels le joueur peut dépenser de l'argent réel pour acquérir des ajouts cosmétiques ou pour simplifier sa progression.

Enfin, on retrouve les grands genres classiques du jeu vidéo : "Simulation" (Simulation de courses automobiles, de vol en avion, de vie), "Strategy", "RPG" (jeux de rôle) et "sports"

### 8.3 Poids des jeux "indie" dans les sorties de chaque année

In [0]:
# 1) Une ligne par jeu / genre
df_genres = (
    df.filter((F.col("release_year") >= 2012) & (F.col("release_year") <= 2022))
      .filter(F.col("release_year").isNotNull())
      .withColumn("genre", F.explode(F.split(F.col("genre"), ",")))
      .withColumn("genre", F.trim(F.col("genre")))
      .filter(F.col("genre") != "")
      .select("appid", "release_year", "genre")
      .dropDuplicates(["appid", "release_year", "genre"])
)

# 2) Nombre total de jeux par année
df_total_games_year = (
    df.filter((F.col("release_year") >= 2012) & (F.col("release_year") <= 2022))
      .filter(F.col("release_year").isNotNull())
      .groupBy("release_year")
      .agg(F.countDistinct("appid").alias("n_games_total"))
)

# 3) Nombre de jeux "Indie" par année
df_indie_games_year = (
    df_genres
    .filter(F.col("genre") == "Indie")
    .groupBy("release_year")
    .agg(F.countDistinct("appid").alias("n_games_indie"))
)

# 4) Jointure + proportion
df_indie_share = (
    df_total_games_year
    .join(df_indie_games_year, on="release_year", how="left")
    .na.fill(0, subset=["n_games_indie"])
    .withColumn(
        "indie_share_pct",
        F.round((F.col("n_games_indie") / F.col("n_games_total")) * 100, 2)
    )
    .orderBy("release_year")
)

display(df_indie_share)

release_year,n_games_total,n_games_indie,indie_share_pct
2012,345,173,50.14
2013,471,262,55.63
2014,1557,870,55.88
2015,2575,1847,71.73
2016,4185,2938,70.2
2017,6017,4263,70.85
2018,7678,5832,75.96
2019,6968,5266,75.57
2020,8305,6234,75.06
2021,8823,6363,72.12


Databricks visualization. Run in Databricks to view.

### 8.4 Fréquence d'association d'autres genres à "Indie"

In [0]:
# Jeux contenant le genre Indie
indie_games = (
    df_genres
    .filter(F.col("genre") == "Indie")
    .select("appid")
    .distinct()
)

# Nombre total de jeux Indie
n_indie = indie_games.count()

# Genres associés aux jeux Indie
indie_genres = (
    df_genres
    .join(indie_games, on="appid", how="inner")
    .filter(F.col("genre") != "Indie")
)

# Comptage des genres associés
genres_associes = (
    indie_genres
    .groupBy("genre")
    .agg(F.countDistinct("appid").alias("n_games"))
)

# Proportion parmi les jeux Indie
genres_associes = genres_associes.withColumn(
    "proportion_indie",
    (F.col("n_games") / F.lit(n_indie)*100)
)

genres_associes = genres_associes.orderBy(F.desc("proportion_indie"))

display(
    genres_associes
    .orderBy(F.desc("proportion_indie"))
    .limit(10)
)

genre,n_games,proportion_indie
Action,18242,46.32888888888888
Casual,17092,43.408253968253966
Adventure,16558,42.05206349206349
Strategy,7766,19.723174603174602
Simulation,7645,19.415873015873018
RPG,7101,18.034285714285716
Early Access,4835,12.279365079365078
Free to Play,2267,5.757460317460318
Sports,1712,4.347936507936508
Racing,1439,3.6546031746031744


Databricks visualization. Run in Databricks to view.

Le graphique ci-dessus présente, parmis tous les jeux ayant le genre "Indie" associé, les 10 genres les plus couramment associés.

On retrouve donc, pour 46% des jeux avec le genre "Indie" rattaché, également le genre "Action". Pour 43%, le genre "Casual" et pour 42%, le genre "Adventure". Ce sont les 3 genres les plus fréquemment rattachés à un jeu en association à "Indie". Viennent ensuite "Strategy" (20%), "Simulation" (19%) et "RPG" (18%)

Cela montre, finalement, que le genre "Indie" constitue une catégorie transversale plutôt qu'un genre au sens strict et se marie avec une grande variété d'autres genres. "Indie" décrit plutôt un mode de production, indiquant qu'un jeu est développé par un studio indépendant

il est important de rappeler que les genres ne sont pas exclusifs : un même jeu peut être associé à plusieurs genres. Les proportions observées ne décrivent donc pas une partition du catalogue mais plutôt la fréquence d’apparition de chaque genre parmi les jeux disponibles. Cette caractéristique limite l’interprétation des fréquences individuelles et invite à analyser plus précisément la manière dont les genres se combinent entre eux, ce qui sera exploré dans le notebook suivant.

# 9. Synthèse de la structure du catalogue Steam

L'analyse du catalogue Steam met en exergue plusieurs caractéristiques.

Tout d’abord, le nombre de jeux publiés augmente fortement au fil du temps, ce qui traduit une croissance rapide du catalogue et une ouverture progressive de la plateforme à un nombre croissant de développeurs. Cette expansion s’accompagne d’une fragmentation marquée des éditeurs : si quelques studios publient de nombreux titres, la majorité des éditeurs ne proposent qu’un nombre limité de jeux.

Sur le plan technique, le catalogue reste très largement centré sur l’écosystème Windows, tandis que les versions Mac et Linux apparaissent beaucoup plus minoritaires.

Par ailleurs, L’analyse des catégories de gameplay montre également que le mode Single-player reste très largement représenté dans le catalogue, même si de nombreuses productions intègrent également des fonctionnalités multijoueur. 

Enfin, le tag Indie domine très nettement la distribution des genres, ce qui reflète la forte présence des productions indépendantes sur la plateforme. Cependant, malgré cette domination, le catalogue conserve une grande diversité de genres, couvrant un large éventail de types de jeux et de styles de gameplay.

Dans l’ensemble, ces éléments décrivent un catalogue très étendu, ouvert à une grande variété de studios et de genres, mais fortement marqué par la place centrale des productions indépendantes et par la prédominance de l’écosystème Windows.

Après cette description de la structure du catalogue Steam, le second notebook s’intéresse aux relations entre genres et à plusieurs dimensions économiques et de réception des jeux.